## Comparación de modelos

Ahora que sabemos qué columnas vamos a utilizar como características, podemos comenzar a probar el rendimiento de diferentes modelos.

Para ello, diseñamos una *pipeline* para la ejecución de los experimentos. Para cada iteración entrenamos y evaluamos a cada modelo y guardamos sus métricas. Una vez han acabdo las iteraciones, cada métrica se agrega en forma de media y desviación estándar para una comparación más representativa.

Los parámetros de estos experimentos los controlaremos a través de un archivo *.yaml*. Contiene los siguientes parámetros:
* *test_size*: Tamaño del conjunto de test
* *n_per_class*: Controla el número de instancias de cada clase que se coge aleatoriamente (si no hay suficientes, se coge el número máximo que exista en el dataset).
* *n_iterations*: Número de iteraciones (train_test_splits + métricas).
* *to_drop*: Características a eliminar
* *to_scale*: Características a escalar 
* *target*: Columna a predecir

Los modelos seleccionados para la comparación son:
* **LogisticRegression**
* **LinearSVC**
* **RandomForest**
* **XGBoost**
* **LightGBM**
* **KNN**

Las métricas utilizadas incluyen *Accuracy*, *Precision*, *Recall* y *Macro-F1*. Adicionalmente, se calculan las métricas de *precision*, *recall* y *f1* para cada clase.

El script que se encarga de aplicar la pipeline se encuentra en **src/comparison_pipeline.py** y los archivos de configuración .yaml se encuentran en la carpeta **src/experiment_config**.

Puede ejecutarse de la siguiente manera:

```bash
py -m src.pipeline balanced_small.yaml

Los resultados son almacenados en formato .json en la carpeta **results/metrics**.

Mediante el script de **src/plots.py** se extraen los resultados del archivos .json y se crean los siguientes plots.

Para esta comparación inicial, utilizaremos un subset balanceado de 15000 instancias por cada clase (menos de "BruteForce", que solo tiene 10000). Calculamos las métricas de cada modelo sobre 20 iteraciones. 
* Todos los parámetros de este experimento están en el archivo **compare_15k.yaml**.

Primero, comparamos las métricas globales.

<img src="../results/plots/compare_15k_macro_metrics.png" width="900">

Con el primer vistazo, claramente podemos ver que los modelos lineales sufren bastante en comparación al resto (diferencia de 0.1 en casi todas las métricas).
* K-NN también presenta un rendimiento inferior a los modelos basados en características.

RandomForest, XGBoost y LightGBM presentan un rendimiento comparable, con los dos últimos siendo prácticamente iguales.
* Para una evaluación inicial con los parámetros default presentan un rendimiento decente (0.72 accuracy y f1).

Ahora vamos a observar el rendimiento para cada clase.

<img src="../results/plots/compare_15k_per_class_f1.png" width="900">

Todos los modelos detectan casi perfectamente los ataques de la clase "Mirai", mientras que en los ataque "Web-based" o "BruteForce" presentan un rendimiento mucho más inferior.
* Los modelos lineales y KNN sufren especialmente para detectar ataques como "Spoofing" en comparación a los modelos basados en características, sugiriendo que no presentan relaciones lineales.
* Esta diferencia de rendimiento se repite para todos los ataques menos "Mirai", "DDoS" y "DoS".

Con esto podemos intuir que los ataques Mirai Y DoS tienen patrones claramente diferenciables respecto al resto de ataques. Además, el tráfico "Benign" y el ataque de tipo "Spoofing" también parecen más diferenciables para los modelos basados en características.

## Análisis de modelo y características

Con estos resultados, elegimos XGBoost como el modelo más apropiado (también podríamos elegir LightGBM). Vamos a evaluar a fondo sus predicciones.
* Todos los parámetros de este experimento están en el archivo **analysis_15k.yaml**. El script utilizado para conseguir los resultados es **src/analysis_pipeline.py**

Comenzamos con la matriz de confusión para todas las clases.

<img src="../results/plots/analysis_15k_XGBoost_confusion_matrix.png" width="550">

Podemos ver varias cosas:
* Los ataques "DDoS" tienden a confundirse con los ataques "DoS". Esto es normal, ya que los ataques comparten patrones similares.
* Los ataques "Mirai" son muy distinguibles.
* Los ataques de tipo "BruteForce", "Recon", "Spoofing", y "Web-based" tienden a confundirse con tráfico "Benign" (0.15, 0.14, 0.12 y 0.11).
    * Tiene sentido ya que muchos de estos ataques no tienen un protocolo distintivo, sino que los paquetes suelen llevar "lo mismo" que llevaría un paquete de tráfico normal.

Ahora vamos a ver cómo asigna nuestro modelo el valor a cada una de las características.

<img src="../results/plots/analysis_15k_XGBoost_feature_importance.png" width="550">

La importancia es dominada claramente por una de las columnas a las que hemos aplicado one-hot encoding, concretamente es la que categoriza si el "Protocol Type" es 47.

El número 47 corresponde al protocolo "GRE", que permite encapsular otros protocolos dentro de paquetes IPv4.

Sin embargo, que el modelo solo se este fijando práticamente en una característica no es muy normal. Vamos a probar cómo funciona si la quitamos del dataset.

<div style="display: flex; gap: 20px;">
    <img src="../results/plots/analysis_15k_no_protocol_type_XGBoost_confusion_matrix.png" width="500">
    <img src="../results/plots/analysis_15k_no_protocol_type_XGBoost_feature_importance.png" width="500">
</div>

Vemos que eliminar "Protocol Type" no ha deteriorado el rendimiento del modelo (la matriz de confusión es muy similar), pero ahora está dando más importancia a un número mayor de características.
* En este caso, la más importante se vuelve el número de paquetes ("Number"), lo cuál tiene sentido, ya que ataques como DoS suele estar asociado a un mayor número de paquetes.

Actualmente quedan 24 features en el dataset. En el script, también evaluamos la importancia de las 4 features que no han salido en el diagrama que muestra la importancia de las top 20 features. Estas son sus importancias:
* **IGMP**: 0.003707
* **IRC**: 0.003087
* **SMTP**: 0.001866
* **Telnet**: 0.000000

**Telnet** es completamente irrelevante y **SMPT** prácticamente también, podemos eliminar estas 2 features y nos quedamos con 22 features.

Antes de eliminar más features, vamos a evaluar su relevancia conforme aumentamos el número de instancias por clase (por si cambia). Al mismo tiempo, vamos a evaluar la eficacia del modelo conforme aumentamos ese número de instancias (lo que también va a provocar que el problema se vuelva progresivamente más desbalanceado).

Para esto, vamos a utilizar un gráfico que muestra cómo la curva de aprendizaje conforme el número de instancias aumenta. Ahora vamos a ejecutar los experimentos con la configuración de **src/experiment_config/analysis_100k_final_features.yaml** que tiene 100000 ejemplos por clase.

<div style="display: flex; gap: 20px;">
    <img src="../results/plots/analysis_100k_final_features_XGBoost_confusion_matrix.png" width="500">
    <img src="../results/plots/analysis_100k_final_features_XGBoost_learning_curve.png" width="500">
</div>

Buenas y malas noticias. La buena es que el rendimiento sigue aumentando con más datos (la línea naranja continúa ascendiendo). La mala viene en los resultados exactos:
* Muchas clases han aumentado su rendimiento ("Benign", "DDoS", "DoS", "Recon", "Spoofing").
* Sin embargo, tanto "Bruteforce" y "Web-based" han sufrido un grán bajón debido al imbalanceo (eran las clases que menos instancias tenían y se quedan muy lejos de 100000, teniendo 10000 y 20000 respectivamente).

Como el resto siguen mejorando, vamos a probar a añadir "sample_weights" al entrenamiento del modelo. Calculamos los pesos de la siguiente manera:

In [ ]:
# No ejecutar, simplemente es un trozo de código del script analysis_pipeline.py
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
model.fit(_x_train, y_train, sample_weight=sample_weights)

Mediante el parámetro "class_weight='balanced'", le decimos que asigne pesos de manera inversamente proporcional al número de samples que haya en el conjunto de entrenamiento. Comprobamos si hemos conseguido remediar el problema de imbalanceo.

<div style="display: flex; gap: 20px;">
    <img src="../results/plots/analysis_100k_final_features_balanced_weights_XGBoost_confusion_matrix.png" width="500">
    <img src="../results/plots/analysis_100k_final_features_balanced_weights_XGBoost_learning_curve.png" width="500">
</div>

Honestamente no ha funcionado. Aunque vuelva a ser capaz de predecir bien "BruteForce" y "Web-based" las mejoras que habíamos visto en las otras categorías (como Recon: 0.77 ha vuelto a bajar a 0.62).
* El rendimiento (aunque haya mejorado un poco como se ve en la curva) es extremadamente similar al rendmiento que habíamos sacado realizando el entrenamiento con 15k por clase (ahora estamos haciendo 100k por clase).
* Por otra parte, como vamos a volver al subset anterior, vamos a eliminar aquellas 4 features que no entraban en el top 20 definitivamente.

Debido a que aumentar el múmero de ejemplos no mejorará el rendimiento de manera significativa, vamos a buscar otra manera de mejorar el funcionamiento. 

En el siguiente paso, realizaremos a la parametrización del modelo ya que hasta ahora hemos estado utilizando los parámetros por defecto del modelo XGBoost.